# Apple background removal (Kaggle)
**Before running:** right-hand settings -> Accelerator = GPU T4, Internet = On, and Add Input -> your raw photos dataset.

Order: run the cells down to the **pilot** image, look at it, set `METHOD` in the "Full run" cell, then run that cell and the ones after it.
Originals are never touched. Results go to `/kaggle/working/output` and `/kaggle/working/output.zip`.

**Fingertip cleanup:** after the background is removed, a second step trims skin-coloured fingertip bumps off the apple's edge
(it learns the hand's colour from the pixels around the apple in each photo). In the pilot, **red = pixels the cleanup removed**.
Set `REMOVE_FINGERTIPS = False` in the config cell to switch it off.

In [ ]:
# 1. Install (about 1-2 minutes). If it complains about numpy, restart the session and run this cell again.
!pip install -q "rembg[gpu]" opencv-python-headless scipy tqdm

In [ ]:
# 2. Imports and config
import os, glob, hashlib, random, shutil
import numpy as np, cv2, torch
from PIL import Image, ImageOps, ImageDraw
from scipy import ndimage as ndi
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

SEARCH_ROOT = "/kaggle/input"
OUT = "/kaggle/working/output"
CUT_DIR   = f"{OUT}/cutouts_rgba"
GRAY_DIR  = f"{OUT}/processed_gray"
SHEET_DIR = f"{OUT}/contact_sheets"
for d in (CUT_DIR, GRAY_DIR, SHEET_DIR):
    os.makedirs(d, exist_ok=True)

GRAY = 128          # neutral background value
SQUARE = 384        # size of processed_gray images
PAD_FRAC = 0.08     # padding around the apple
CUT_MAX_SIDE = 640  # keeps the RGBA cutouts (and your download) small
REMOVE_FINGERTIPS = True   # trim skin-coloured fingertip bumps off the mask
SEED = 42
random.seed(SEED); np.random.seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

In [ ]:
# 3. Find the images and skip exact duplicates
EXT = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
paths = sorted(p for p in glob.glob(f"{SEARCH_ROOT}/**/*", recursive=True)
               if os.path.splitext(p)[1].lower() in EXT)
print("images found:", len(paths))

seen, unique, dups = {}, [], []
for p in paths:
    h = hashlib.md5(open(p, "rb").read()).hexdigest()
    if h in seen:
        dups.append((p, seen[h]))
    else:
        seen[h] = p
        unique.append(p)
print("unique:", len(unique), "| exact duplicates skipped:", len(dups))

# Unique output names (same stem as the original; a counter is added only if two stems clash)
names, used = {}, set()
for p in unique:
    stem = os.path.splitext(os.path.basename(p))[0]
    n, i = stem, 1
    while n in used:
        i += 1; n = f"{stem}_{i}"
    used.add(n); names[p] = n

def load(p):
    return ImageOps.exif_transpose(Image.open(p)).convert("RGB")

In [ ]:
# 3b. Fingertip cleanup (trims skin-coloured fingertip bumps from the apple mask)
import numpy as np, cv2
from scipy import ndimage as ndi


def _lab(rgb):
    # float Lab: L 0..100, a/b about -100..100
    return cv2.cvtColor(rgb.astype(np.float32) / 255.0, cv2.COLOR_RGB2LAB)


def _hue_chroma(lab):
    a, b = lab[..., 1], lab[..., 2]
    return np.degrees(np.arctan2(b, a)) % 360, np.hypot(a, b)


def remove_fingers(rgb, mask, band=0.50, max_removed=0.22, debug=False, hand_from_photo=True):
    """Trim skin-coloured bumps (fingertips) that touch the outer edge of an apple mask.

    rgb  : HxWx3 uint8 photo (the ORIGINAL photo, so the hand outside the mask can be sampled)
    mask : HxW bool/uint8 apple mask (may include fingertips)
    Returns (new_mask bool, info dict). Never touches pixels deep inside the apple (defects stay).
    """
    m = mask > 0 if mask.dtype != bool else mask.copy()
    info = dict(removed=0.0, hand_source="none", skipped="")
    area = int(m.sum())
    if area < 200:
        info["skipped"] = "tiny_mask"
        return m, info

    lab = _lab(rgb)
    hue, chroma = _hue_chroma(lab)
    L = lab[..., 0]

    dist = cv2.distanceTransform(m.astype(np.uint8), cv2.DIST_L2, 5)
    dmax = float(dist.max())
    core = dist > 0.5 * dmax
    if core.sum() < 50:
        info["skipped"] = "no_core"
        return m, info

    # apple colour (robust, defects barely move a median)
    a_med = float(np.median(lab[..., 1][core]))
    b_med = float(np.median(lab[..., 2][core]))
    L_med = float(np.median(L[core]))

    # ---- hand colour ---------------------------------------------------
    # prior: skin hue band. Used to find hand pixels OUTSIDE the mask (the palm next to the apple).
    prior = (hue > 25) & (hue < 85) & (chroma > 7) & (chroma < 50) & (L > 35)
    ring = cv2.dilate(m.astype(np.uint8), cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE, (int(dmax * 1.2) | 1, int(dmax * 1.2) | 1))).astype(bool) & ~m
    outside = prior & ring
    # keep only skin-like blobs that actually touch the apple (so feet / wood are ignored)
    touch = cv2.dilate(m.astype(np.uint8), np.ones((9, 9), np.uint8)).astype(bool)
    lab_o, n_o = ndi.label(outside)
    if n_o:
        keep_ids = np.unique(lab_o[touch & outside])
        keep_ids = keep_ids[keep_ids > 0]
        outside = np.isin(lab_o, keep_ids)
    if hand_from_photo and outside.sum() > 400:
        h_a = float(np.median(lab[..., 1][outside]))
        h_b = float(np.median(lab[..., 2][outside]))
        h_L = float(np.median(L[outside]))
        info["hand_source"] = "photo"
        info["hand_px"] = int(outside.sum())
        info["hand_lab"] = (round(h_L, 1), round(h_a, 1), round(h_b, 1))
    else:
        h_a, h_b, h_L = 12.0, 18.0, 75.0     # generic fallback
        info["hand_source"] = "prior"

    # ---- classify mask pixels ------------------------------------------
    da = np.hypot(lab[..., 1] - a_med, lab[..., 2] - b_med)   # distance to apple colour (chroma plane)
    dh = np.hypot(lab[..., 1] - h_a, lab[..., 2] - h_b)       # distance to hand colour
    in_band = m & (dist <= band * dmax)
    strong = prior & (dh < 9.0) & (dh < 0.6 * da) & (np.abs(L - h_L) < 28) & in_band
    # weak = shadowed / blended hand pixels; only kept if they are connected to strong ones
    weak = ((hue > 20) & (hue < 95) & (chroma > 5) & (dh < 16.0) & (dh < 0.85 * da)
            & (np.abs(L - h_L) < 40) & in_band) | strong

    # join speckle, drop dust
    k3 = np.ones((3, 3), np.uint8)
    strong = cv2.morphologyEx(strong.astype(np.uint8), cv2.MORPH_CLOSE, k3).astype(bool) & m
    weak = cv2.morphologyEx(weak.astype(np.uint8), cv2.MORPH_CLOSE, k3).astype(bool) & m
    lab_w, n_w = ndi.label(weak)
    removed = np.zeros_like(m)
    min_area = max(30, int(0.002 * area))
    # A fingertip is a THICK blob. Thin pinkish rims along the apple's edge (blush, shading) are not.
    ro = max(2, int(round(0.05 * dmax)))
    disk = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * ro + 1, 2 * ro + 1))
    for i in range(1, n_w + 1):
        comp = lab_w == i
        if (comp & strong).sum() < min_area:      # needs a solid skin-coloured core
            continue
        if dist[comp].min() > 4:                  # must touch the outer edge: interior stuff is never removed
            continue
        thick = cv2.morphologyEx(comp.astype(np.uint8), cv2.MORPH_OPEN, disk).astype(bool)
        if thick.sum() < min_area:                # only thin slivers -> leave them alone
            continue
        # regrow a little so the bump's own edge is included, but not the thin tails
        removed |= cv2.dilate(thick.astype(np.uint8), disk).astype(bool) & comp

    if not removed.any():
        return (m, info) if not debug else (m, info, removed)

    # take the blurry transition pixels around the bump too
    r = max(2, int(round(0.012 * np.sqrt(area))))
    fringe = cv2.dilate(removed.astype(np.uint8), cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE, (2 * r + 1, 2 * r + 1))).astype(bool)
    # ...but only the fringe that is not clearly apple-coloured
    fringe &= (da > 6.0) | removed
    removed |= fringe & m

    frac = removed.sum() / area
    if frac > max_removed:
        info["skipped"] = f"would_remove_{frac:.0%}"
        return (m, info) if not debug else (m, info, removed)

    new = m & ~removed
    # tidy: keep the biggest piece, fill holes, smooth the cut a little
    new = ndi.binary_fill_holes(new)
    k = max(3, int(round(0.01 * np.sqrt(area))) | 1)
    new = cv2.morphologyEx(new.astype(np.uint8), cv2.MORPH_OPEN,
                           cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))).astype(bool)
    lab_n, n_n = ndi.label(new)
    if n_n > 1:
        sizes = ndi.sum(new, lab_n, range(1, n_n + 1))
        new = lab_n == (int(np.argmax(sizes)) + 1)
    info["removed"] = float(1 - new.sum() / area)
    return (new, info) if not debug else (new, info, removed)


def clean_mask(rgb, mask, passes=2, **kw):
    """Run remove_fingers up to `passes` times (a second pass catches thin leftovers)."""
    m = mask > 0 if mask.dtype != bool else mask.copy()
    total, info = 0.0, dict(removed=0.0, hand_source="none", skipped="")
    for _ in range(passes):
        m2, inf = remove_fingers(rgb, m, **kw)
        info["hand_source"] = inf["hand_source"]
        if inf["skipped"]:
            info["skipped"] = inf["skipped"]
        if inf["removed"] <= 0:
            break
        m = m2
        total += inf["removed"]
    info["removed"] = total
    return m, info

In [ ]:
# 4. Segmentation methods + pilot comparison
from rembg import remove, new_session

sessions = {}
def get_rembg(name):
    if name not in sessions:
        sessions[name] = new_session(name)
    return sessions[name]

def mask_rembg(img, model="isnet-general-use"):
    return np.array(remove(img, session=get_rembg(model), only_mask=True))

# --- GroundingDINO (finds the apple box) + SAM (cuts it out) via transformers ---
_ds = {}
def load_dino_sam():
    if _ds: return
    from transformers import (AutoProcessor, AutoModelForZeroShotObjectDetection,
                              SamModel, SamProcessor)
    _ds["dp"] = AutoProcessor.from_pretrained("IDEA-Research/grounding-dino-tiny")
    _ds["dm"] = AutoModelForZeroShotObjectDetection.from_pretrained(
        "IDEA-Research/grounding-dino-tiny").to(DEVICE).eval()
    _ds["sp"] = SamProcessor.from_pretrained("facebook/sam-vit-base")
    _ds["sm"] = SamModel.from_pretrained("facebook/sam-vit-base").to(DEVICE).eval()

def detect_box(img):
    load_dino_sam()
    dp, dm = _ds["dp"], _ds["dm"]
    inputs = dp(images=img, text="an apple.", return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = dm(**inputs)
    kw = dict(text_threshold=0.2, target_sizes=[img.size[::-1]])
    try:
        res = dp.post_process_grounded_object_detection(out, inputs.input_ids, threshold=0.25, **kw)[0]
    except TypeError:  # older transformers versions
        res = dp.post_process_grounded_object_detection(out, inputs.input_ids, box_threshold=0.25, **kw)[0]
    W, H = img.size
    best, best_s = None, -1
    for b, s in zip(res["boxes"].tolist(), res["scores"].tolist()):
        area = (b[2]-b[0]) * (b[3]-b[1]) / (W*H)
        if area < 0.9 and s > best_s:   # ignore boxes that cover the whole photo
            best, best_s = b, s
    return best

def mask_sam(img, box=None):
    box = box or detect_box(img)
    if box is None:
        return None
    sp, sm = _ds["sp"], _ds["sm"]
    inputs = sp(img, input_boxes=[[box]], return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = sm(**inputs, multimask_output=False)
    m = sp.image_processor.post_process_masks(
        out.pred_masks.cpu(), inputs["original_sizes"].cpu(),
        inputs["reshaped_input_sizes"].cpu())[0]
    return (m[0, 0].numpy().astype(np.uint8)) * 255

def mask_hybrid(img):
    # rembg mask, but only inside the detector's box (helps cut off the hand)
    m = mask_rembg(img)
    box = detect_box(img)
    if box is None:
        return m
    W, H = img.size
    bw, bh = box[2]-box[0], box[3]-box[1]
    x0 = max(0, int(box[0]-0.03*bw)); y0 = max(0, int(box[1]-0.03*bh))
    x1 = min(W, int(box[2]+0.03*bw)); y1 = min(H, int(box[3]+0.03*bh))
    keep = np.zeros_like(m); keep[y0:y1, x0:x1] = 1
    return m * keep

METHODS = {
    "rembg_isnet": lambda im: mask_rembg(im, "isnet-general-use"),
    "dino+sam":    mask_sam,
    "hybrid":      mask_hybrid,
}
try:
    get_rembg("birefnet-general")
    METHODS["rembg_birefnet"] = lambda im: mask_rembg(im, "birefnet-general")
except Exception as e:
    print("birefnet not available, skipping:", e)

def main_blob(mask):
    # biggest connected piece of a 0-255 mask, holes filled -> bool array (or None)
    if mask is None:
        return None
    b = mask > 127
    lab, n = ndi.label(b)
    if n == 0:
        return None
    sizes = ndi.sum(b, lab, range(1, n+1))
    return ndi.binary_fill_holes(lab == (int(np.argmax(sizes)) + 1))

def preview(img, keep, removed=None):
    # apple on gray; pixels removed by the fingertip cleanup are painted red
    rgb = np.array(img).copy()
    out = np.where(keep[..., None], rgb, GRAY).astype(np.uint8)
    if removed is not None:
        out[removed] = (255, 0, 0)
    return Image.fromarray(out)

# Pilot: 12 random images, every method side by side (fingertip cleanup already applied; red = removed by it)
pilot = random.sample(unique, min(12, len(unique)))
cols = ["original"] + list(METHODS)
fig, axes = plt.subplots(len(pilot), len(cols), figsize=(3*len(cols), 3*len(pilot)))
for r, p in enumerate(pilot):
    img = load(p); img.thumbnail((512, 512))
    axes[r][0].imshow(img)
    for c, (mname, fn) in enumerate(METHODS.items(), start=1):
        try:
            b = main_blob(fn(img)) if fn is not None else None
            if b is None:
                axes[r][c].imshow(np.full((10, 10, 3), 255)); continue
            removed = None
            if REMOVE_FINGERTIPS:
                b2, finfo = clean_mask(np.array(img), b)
                removed = b & ~b2
                b = b2
            axes[r][c].imshow(preview(img, b, removed))
        except Exception as e:
            axes[r][c].text(0.1, 0.5, "error"); print(mname, p, e)
for r in range(len(pilot)):
    for c in range(len(cols)):
        axes[r][c].axis("off")
        if r == 0: axes[r][c].set_title(cols[c])
plt.tight_layout()
plt.savefig(f"{OUT}/pilot.jpg", dpi=70)
plt.show()

**Look at the pilot above** (red = removed by the fingertip cleanup). Things to check:
1. Fingers or fingertips still attached to the apple. Try `hybrid` or `dino+sam`.
2. The brown defect cut out of the apple. That ruins the data, so pick another method.
3. Red on real apple surface (not on a finger). If the cleanup eats apple edge, set `REMOVE_FINGERTIPS = False` and tell me.

Then set `METHOD` in the next cell.

In [ ]:
# 5. Full run. Set METHOD from the pilot: "rembg_isnet", "rembg_birefnet", "dino+sam" or "hybrid"
METHOD = "hybrid"
seg = METHODS[METHOD]

def postprocess(mask, rgb=None):
    # returns (alpha uint8 with a soft edge, binary mask, info for flags)
    if mask is None:
        return None, None, dict(area=0, big=0, sides=0, reason="no_detection")
    b = mask > 127
    lab, n = ndi.label(b)
    if n == 0:
        return None, None, dict(area=0, big=0, sides=0, reason="empty_mask")
    sizes = ndi.sum(b, lab, range(1, n+1))
    main = int(np.argmax(sizes)) + 1
    big = int((sizes > 0.10 * sizes.max()).sum())   # number of large components
    b = ndi.binary_fill_holes(lab == main)
    finfo = dict(removed=0.0, skipped="")
    if REMOVE_FINGERTIPS and rgb is not None:
        b, finfo = clean_mask(rgb, b)               # trim fingertip bumps (uses the ORIGINAL photo's colours)
    H, W = b.shape
    sides = int(b[0].sum() > 3) + int(b[-1].sum() > 3) + int(b[:, 0].sum() > 3) + int(b[:, -1].sum() > 3)
    alpha = cv2.GaussianBlur(b.astype(np.float32), (0, 0), 1.0)
    alpha = (np.clip(alpha, 0, 1) * 255).astype(np.uint8)
    return alpha, b, dict(area=float(b.mean()), big=big, sides=sides, reason="",
                          fingers=finfo["removed"], fskip=finfo["skipped"])

def flag_reasons(info):
    r = [info["reason"]] if info["reason"] else []
    if info["area"] and info["area"] < 0.05: r.append("tiny_mask")
    if info["area"] > 0.80: r.append("huge_mask")
    if info["big"] > 1: r.append("several_components")
    if info["sides"] >= 2: r.append("touches_borders")
    if info.get("fskip"): r.append("fingertip_cleanup_skipped:" + info["fskip"])   # check this one by eye
    if info.get("fingers", 0) > 0.12: r.append("cleanup_removed_a_lot")            # check this one by eye
    return r

def square_gray(rgb, a):
    comp = rgb.astype(np.float32) * (a[..., None]/255.0) + GRAY * (1 - a[..., None]/255.0)
    im = Image.fromarray(comp.astype(np.uint8))
    s = SQUARE / max(im.size)
    im = im.resize((max(1, round(im.width*s)), max(1, round(im.height*s))), Image.LANCZOS)
    canvas = Image.new("RGB", (SQUARE, SQUARE), (GRAY, GRAY, GRAY))
    canvas.paste(im, ((SQUARE-im.width)//2, (SQUARE-im.height)//2))
    return canvas

rows = []   # (name, flags)
cleaned = []   # images where the fingertip cleanup removed something
for p in tqdm(unique):
    name = names[p]
    try:
        img = load(p)
        rgb = np.array(img)
        alpha, b, info = postprocess(seg(img), rgb)
        flags = flag_reasons(info)
        if alpha is None:
            rows.append((name, flags)); continue
        if info.get("fingers", 0) > 0:
            cleaned.append(name)
        ys, xs = np.where(b)
        x0, x1, y0, y1 = xs.min(), xs.max()+1, ys.min(), ys.max()+1
        pad = int(PAD_FRAC * max(x1-x0, y1-y0))
        x0, y0 = max(0, x0-pad), max(0, y0-pad)
        x1, y1 = min(img.width, x1+pad), min(img.height, y1+pad)
        crop_rgb, crop_a = rgb[y0:y1, x0:x1], alpha[y0:y1, x0:x1]

        rgba = Image.fromarray(np.dstack([crop_rgb, crop_a]), "RGBA")
        if max(rgba.size) > CUT_MAX_SIDE:
            s = CUT_MAX_SIDE / max(rgba.size)
            rgba = rgba.resize((round(rgba.width*s), round(rgba.height*s)), Image.LANCZOS)
        rgba.save(f"{CUT_DIR}/{name}.png", optimize=True)
        square_gray(crop_rgb, crop_a).save(f"{GRAY_DIR}/{name}.jpg", quality=95)
        rows.append((name, flags))
    except Exception as e:
        rows.append((name, [f"error: {e}"]))

flagged = [(n, f) for n, f in rows if f]
failed  = [(n, f) for n, f in rows if any(x.startswith(("no_detection", "empty_mask", "error")) for x in f)]
print(f"processed: {len(rows)-len(failed)} | flagged: {len(flagged)} | failed: {len(failed)}")

In [ ]:
# 6. Contact sheets (original | result), all images, plus flagged images separately
src_by_name = {names[p]: p for p in unique}
T = 190  # thumbnail size

def pair(name):
    o = load(src_by_name[name]); o.thumbnail((T, T))
    a = Image.new("RGB", (T, T), (255, 255, 255)); a.paste(o, ((T-o.width)//2, (T-o.height)//2))
    g_path = f"{GRAY_DIR}/{name}.jpg"
    g = Image.open(g_path).resize((T, T)) if os.path.exists(g_path) else Image.new("RGB", (T, T), (200, 0, 0))
    return a, g

def make_sheets(items, prefix, per_sheet=25, cols=5):
    for s in range(0, len(items), per_sheet):
        chunk = items[s:s+per_sheet]
        rws = (len(chunk) + cols - 1) // cols
        sheet = Image.new("RGB", (cols*2*T, rws*(T+14)), (255, 255, 255))
        d = ImageDraw.Draw(sheet)
        for i, name in enumerate(chunk):
            a, g = pair(name)
            x, y = (i % cols)*2*T, (i // cols)*(T+14)
            sheet.paste(a, (x, y+14)); sheet.paste(g, (x+T, y+14))
            d.text((x+2, y+1), name[:40], fill=(0, 0, 0))
        sheet.save(f"{SHEET_DIR}/{prefix}_{s//per_sheet+1:02d}.jpg", quality=85)

all_names = [n for n, _ in rows]
make_sheets(all_names, "all")
make_sheets([n for n, _ in flagged], "FLAGGED")
make_sheets(cleaned, "CLEANED")   # every image where the fingertip cleanup changed something: check these by eye

with open(f"{OUT}/qc_report.md", "w") as f:
    f.write(f"# QC report\n\nmethod: {METHOD}\nfingertip cleanup: {'on' if REMOVE_FINGERTIPS else 'off'}\n\n")
    f.write(f"images found: {len(paths)}\nexact duplicates skipped: {len(dups)}\n")
    f.write(f"processed OK: {len(rows)-len(failed)}\nflagged: {len(flagged)}\nfailed: {len(failed)}\n")
    f.write(f"fingertip cleanup changed: {len(cleaned)} (see CLEANED_*.jpg)\n\n## Flagged\n")
    for n, fl in flagged:
        f.write(f"- {n}: {', '.join(fl)}\n")
print(open(f"{OUT}/qc_report.md").read())

first = sorted(glob.glob(f"{SHEET_DIR}/all_*.jpg"))[0]
plt.figure(figsize=(16, 9)); plt.imshow(Image.open(first)); plt.axis("off"); plt.show()

In [ ]:
# 7. Zip everything for download (find it in the Output tab / right-hand file panel)
shutil.make_archive("/kaggle/working/output", "zip", OUT)
print("size MB:", round(os.path.getsize("/kaggle/working/output.zip")/1e6, 1))